# Baseline Comparisons & Post-hoc Analyses

This notebook implements the following items from the manuscript improvement plan:

- **E3**: Frozen AST Baseline (Linear Probing) — answers "is fine-tuning needed?"
- **E4**: Logistic Regression Baseline — answers "is deep learning needed?"
- **E5**: Comparison Table (fine-tuned AST vs. frozen AST vs. logistic regression)
- **R1**: Calibration Analysis (all three conditions)
- **R2**: Per-Task Contribution Analysis (PD)
- **R3**: Comorbidity Quantification (PD/Dementia overlap)

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TORCH"] = "1"

import torch, random, numpy as np
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 1. Data Loading & PD Task Selection

Replicates the data pipeline from `PD_AST_selected_tasks_Spectrograms + Metadata_eval.ipynb`.

In [ ]:
#1 - Imports & paths
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path('/data0/b2ai-voice/2.0.0')
SPEC = ROOT / 'spectrogram.parquet'
PHEN = ROOT / 'phenotype.tsv'
print(SPEC.exists(), PHEN.exists())

In [ ]:
#2 - Load files (safe row-group read)
pheno = pd.read_csv(PHEN, sep='\t')

pf = pq.ParquetFile(SPEC)
parts = []
for i in range(pf.num_row_groups):
    parts.append(pf.read_row_group(i, columns=['participant_id','session_id','task_name','spectrogram']).to_pandas())
spec = pd.concat(parts, ignore_index=True)

print("pheno:", pheno.shape, "spec:", spec.shape)

In [ ]:
#3 - Build PD label & merge
pheno['parkinsons_label'] = pheno['parkinsons'].map({'Checked':1, 'Unchecked':0})
labels = pheno[['participant_id','parkinsons_label']].dropna()
labels['parkinsons_label'] = labels['parkinsons_label'].astype(int)

data = spec.merge(labels, on='participant_id', how='inner')
print("merged:", data.shape)

# Compute time_frames
data['time_frames'] = data['spectrogram'].apply(lambda s: np.stack(s).shape[1])
print("PD label counts:", data['parkinsons_label'].value_counts().to_dict())

In [ ]:
#4 - Select HIGH PD-ratio tasks + process spectrograms
from tqdm import tqdm

high_pd_only_tasks = [
    'Cinderella-Story',
    'Productive-Vocabulary-1',
    'Productive-Vocabulary-2',
    'Productive-Vocabulary-3',
    'Productive-Vocabulary-4',
    'Productive-Vocabulary-5',
    'Productive-Vocabulary-6',
    'Word-color-Stroop',
    'Random-Item-Generation',
]

min_time_frames = 100
data_high_pd = data[
    (data['task_name'].isin(high_pd_only_tasks)) &
    (data['time_frames'] >= min_time_frames)
].copy()

print(f"Samples: {len(data_high_pd)}, PD: {data_high_pd['parkinsons_label'].sum()}, "
      f"Non-PD: {(data_high_pd['parkinsons_label']==0).sum()}, "
      f"Ratio: {data_high_pd['parkinsons_label'].mean():.2%}")
print(f"Participants: {data_high_pd['participant_id'].nunique()}")

TARGET_SEQ_LEN = 1024

def process_spectrogram_raw(spec_raw, target_len=1024):
    """Process raw spectrogram with reflect padding."""
    spec = np.stack(spec_raw).astype(np.float32)
    n_mels, time_len = spec.shape
    if time_len < target_len:
        spec = np.pad(spec, ((0, 0), (0, target_len - time_len)), mode='reflect')
    elif time_len > target_len:
        start = (time_len - target_len) // 2
        spec = spec[:, start:start + target_len]
    return spec

X_list = []
for idx, row in tqdm(data_high_pd.iterrows(), total=len(data_high_pd), desc="Processing"):
    X_list.append(process_spectrogram_raw(row['spectrogram'], TARGET_SEQ_LEN))

X_raw = np.stack(X_list)
y_raw = data_high_pd['parkinsons_label'].values
participants_raw = data_high_pd['participant_id'].values
task_names_raw = data_high_pd['task_name'].values

print(f"Processed shape: {X_raw.shape}")

# Participant-level setup for CV
unique_participants = np.unique(participants_raw)
participant_labels = np.array([y_raw[participants_raw == p][0] for p in unique_participants])
print(f"Unique participants: {len(unique_participants)}, PD: {participant_labels.sum()}, Non-PD: {(participant_labels==0).sum()}")

---
## 2. E4: Logistic Regression Baseline

Extract summary statistics from each spectrogram, aggregate to participant level, and train logistic regression with the same 5-fold participant-level CV.

In [ ]:
#5 - Feature extraction from spectrograms
from scipy.stats import skew, kurtosis

def extract_features(spectrogram):
    """Extract summary statistics from a 201x1024 spectrogram."""
    per_bin_mean = spectrogram.mean(axis=1)
    per_bin_std = spectrogram.std(axis=1)
    per_bin_skew = skew(spectrogram, axis=1)
    per_bin_kurt = kurtosis(spectrogram, axis=1)

    freq_weights = spectrogram.mean(axis=1)
    freq_weights_safe = np.clip(freq_weights, 1e-10, None)
    freq_axis = np.arange(spectrogram.shape[0])
    spectral_centroid = np.average(freq_axis, weights=freq_weights_safe)
    spectral_bandwidth = np.sqrt(np.average(
        (freq_axis - spectral_centroid)**2, weights=freq_weights_safe
    ))

    global_feats = np.array([
        spectrogram.mean(),
        spectrogram.std(),
        (spectrogram ** 2).sum(),
        spectral_centroid,
        spectral_bandwidth,
    ])

    feats = np.concatenate([per_bin_mean, per_bin_std, per_bin_skew, per_bin_kurt, global_feats])
    # Replace NaN/Inf from constant-value bins (skew/kurtosis undefined)
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats

print("Extracting features from spectrograms...")
features = np.array([extract_features(x) for x in tqdm(X_raw, desc="Extracting")])
print(f"Feature matrix shape: {features.shape}")
print(f"Features per recording: {features.shape[1]} "
      f"(201 bins x 4 stats = {201*4} + 5 global = {201*4+5})")
print(f"NaN count: {np.isnan(features).sum()}, Inf count: {np.isinf(features).sum()}")

In [ ]:
#6 - Logistic Regression 5-fold CV (participant-level)
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
import time

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

lr_fold_results = []
lr_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
lr_oof_labels = participant_labels.astype(np.int64).copy()

start = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(unique_participants, participant_labels)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    train_parts_fold = unique_participants[train_idx]
    val_parts_fold = unique_participants[val_idx]

    train_mask = np.isin(participants_raw, train_parts_fold)
    val_mask = np.isin(participants_raw, val_parts_fold)

    # Recording-level features
    X_train_rec = features[train_mask]
    y_train_rec = y_raw[train_mask]
    parts_train_rec = participants_raw[train_mask]

    X_val_rec = features[val_mask]
    y_val_rec = y_raw[val_mask]
    parts_val_rec = participants_raw[val_mask]

    # Aggregate to participant level (mean across recordings)
    def aggregate_by_participant(X, y, parts):
        unique_p = np.unique(parts)
        X_agg, y_agg = [], []
        for p in unique_p:
            mask = parts == p
            X_agg.append(X[mask].mean(axis=0))
            y_agg.append(y[mask][0])
        return np.array(X_agg), np.array(y_agg), unique_p

    X_train_part, y_train_part, train_pids = aggregate_by_participant(X_train_rec, y_train_rec, parts_train_rec)
    X_val_part, y_val_part, val_pids = aggregate_by_participant(X_val_rec, y_val_rec, parts_val_rec)

    print(f"  Train: {len(X_train_part)} participants, Val: {len(X_val_part)} participants")

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_part)
    X_val_scaled = scaler.transform(X_val_part)

    # Fit logistic regression
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, C=1.0)
    clf.fit(X_train_scaled, y_train_part)

    # Predict
    val_probs = clf.predict_proba(X_val_scaled)[:, 1]

    # Store OOF predictions
    for i, pid in enumerate(val_pids):
        idx = np.where(unique_participants == pid)[0][0]
        lr_oof_probs[idx] = val_probs[i]

    # Fold metrics
    fold_auc = roc_auc_score(y_val_part, val_probs)
    fpr, tpr, thresholds = roc_curve(y_val_part, val_probs)
    opt_idx = np.argmax(tpr - fpr)
    opt_thresh = thresholds[opt_idx]
    preds_opt = (val_probs >= opt_thresh).astype(int)

    lr_fold_results.append({
        'fold': fold + 1,
        'auc': float(fold_auc),
        'f1': float(f1_score(y_val_part, preds_opt, zero_division=0)),
        'recall': float(recall_score(y_val_part, preds_opt, zero_division=0)),
        'precision': float(precision_score(y_val_part, preds_opt, zero_division=0)),
        'threshold': float(opt_thresh),
    })

    print(f"  AUC={fold_auc:.4f}, F1={lr_fold_results[-1]['f1']:.4f}, "
          f"Rec={lr_fold_results[-1]['recall']:.4f}, Prec={lr_fold_results[-1]['precision']:.4f}")

elapsed = time.time() - start

# Summary
from scipy import stats as sp_stats

print("\n" + "="*60)
print("LOGISTIC REGRESSION BASELINE - Per-fold results")
print("="*60)
for r in lr_fold_results:
    print(f"  Fold {r['fold']}: AUC={r['auc']:.4f}  F1={r['f1']:.4f}  Rec={r['recall']:.4f}  Prec={r['precision']:.4f}")

n = N_FOLDS
t_crit = sp_stats.t.ppf(0.975, df=n-1)

for metric_name in ['auc', 'f1', 'recall', 'precision']:
    vals = [r[metric_name] for r in lr_fold_results]
    m = np.mean(vals)
    sd = np.std(vals, ddof=1)
    se = sd / np.sqrt(n)
    ci_lo, ci_hi = m - t_crit * se, m + t_crit * se
    print(f"  Mean {metric_name.upper()}: {m:.4f} +/- {sd:.4f}  95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")

# OOF metrics
lr_oof_auc = roc_auc_score(lr_oof_labels, lr_oof_probs)
fpr, tpr, thresholds = roc_curve(lr_oof_labels, lr_oof_probs)
opt_idx = np.argmax(tpr - fpr)
lr_oof_thresh = thresholds[opt_idx]
lr_oof_preds = (lr_oof_probs >= lr_oof_thresh).astype(int)

print(f"\nOOF AUC:       {lr_oof_auc:.4f}")
print(f"OOF F1:        {f1_score(lr_oof_labels, lr_oof_preds, zero_division=0):.4f} (threshold={lr_oof_thresh:.3f})")
print(f"OOF Recall:    {recall_score(lr_oof_labels, lr_oof_preds, zero_division=0):.4f}")
print(f"OOF Precision: {precision_score(lr_oof_labels, lr_oof_preds, zero_division=0):.4f}")
print(f"\nTotal time: {elapsed:.1f}s")

# Save
np.savez("lr_baseline_cv_results.npz",
    oof_probs=lr_oof_probs,
    oof_labels=lr_oof_labels,
    participant_ids=unique_participants,
    fold_aucs=np.array([r['auc'] for r in lr_fold_results]),
    fold_f1s=np.array([r['f1'] for r in lr_fold_results]),
    fold_recalls=np.array([r['recall'] for r in lr_fold_results]),
    fold_precisions=np.array([r['precision'] for r in lr_fold_results]),
)
print("Saved: lr_baseline_cv_results.npz")

---
## 3. E3: Frozen AST Baseline (Linear Probing)

Freeze all AST backbone parameters; train only the classification head. Uses same 5-fold participant-level CV with identical splits.

In [ ]:
#7 - AST model definition (reused from PD notebook)
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ASTModel, ASTConfig
from scipy.ndimage import zoom

class ASTClassifier(nn.Module):
    """AST-based classifier using pre-trained weights."""
    def __init__(self, num_classes=2, pretrained=True, freeze_base=False):
        super().__init__()
        if pretrained:
            self.ast = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            hidden_size = self.ast.config.hidden_size
        else:
            config = ASTConfig(
                hidden_size=768, num_hidden_layers=12, num_attention_heads=12,
                intermediate_size=3072, max_length=1024, num_mel_bins=128,
            )
            self.ast = ASTModel(config)
            hidden_size = 768

        if freeze_base:
            for param in self.ast.parameters():
                param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        outputs = self.ast(input_values=x)
        pooled = outputs.pooler_output
        logits = self.classifier(pooled)
        return logits

class ASTDatasetCV(torch.utils.data.Dataset):
    def __init__(self, X, y, participants, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.participants = np.array(participants)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            if np.random.random() < 0.5:
                t = np.random.randint(50, 150)
                t0 = np.random.randint(0, max(1, x.shape[1] - t))
                x[:, t0:t0+t] = 0
            if np.random.random() < 0.5:
                f = np.random.randint(10, 30)
                f0 = np.random.randint(0, max(1, x.shape[0] - f))
                x[f0:f0+f, :] = 0
        return {'inputs': x, 'labels': self.y[idx], 'participant': self.participants[idx]}

class FocalLossCV(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

def resize_spectrogram_cv(spec, target_mel=128, target_time=1024):
    mel_ratio = target_mel / spec.shape[0]
    time_ratio = target_time / spec.shape[1]
    return zoom(spec, (mel_ratio, time_ratio), order=1).astype(np.float32)

def evaluate_fold(model, loader, device):
    model.eval()
    all_probs, all_labels, all_parts = [], [], []
    with torch.no_grad():
        for batch in loader:
            inputs = batch['inputs'].to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
            all_parts.extend(batch['participant'])

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_parts = np.array(all_parts)

    unique_parts = np.unique(all_parts)
    part_probs, part_labels = [], []
    for p in unique_parts:
        mask = all_parts == p
        part_probs.append(all_probs[mask].mean())
        part_labels.append(all_labels[mask][0])

    return np.array(part_probs), np.array(part_labels), unique_parts

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
#8 - Frozen AST 5-fold CV
import copy
import time

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

frozen_fold_results = []
frozen_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
frozen_oof_labels = participant_labels.astype(np.int64).copy()

total_start = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(unique_participants, participant_labels)):
    print(f"\n{'='*60}")
    print(f"Fold {fold+1}/{N_FOLDS} (FROZEN AST - Linear Probe)")
    print(f"{'='*60}")

    train_parts_fold = unique_participants[train_idx]
    val_parts_fold = unique_participants[val_idx]

    train_mask = np.isin(participants_raw, train_parts_fold)
    val_mask = np.isin(participants_raw, val_parts_fold)

    X_train_fold = X_raw[train_mask]
    y_train_fold = y_raw[train_mask]
    parts_train_fold = participants_raw[train_mask]

    X_val_fold = X_raw[val_mask]
    y_val_fold = y_raw[val_mask]
    parts_val_fold = participants_raw[val_mask]

    print(f"Train: {len(X_train_fold)} recordings from {len(train_parts_fold)} participants")
    print(f"Val:   {len(X_val_fold)} recordings from {len(val_parts_fold)} participants")

    # Resize
    X_train_ast_fold = np.stack([resize_spectrogram_cv(x) for x in tqdm(X_train_fold, desc="resize train", leave=False)])
    X_val_ast_fold = np.stack([resize_spectrogram_cv(x) for x in tqdm(X_val_fold, desc="resize val", leave=False)])

    # Normalize on fold-specific training data
    fold_mean = X_train_ast_fold.mean()
    fold_std = X_train_ast_fold.std()
    X_train_ast_fold = (X_train_ast_fold - fold_mean) / (fold_std + 1e-8)
    X_val_ast_fold = (X_val_ast_fold - fold_mean) / (fold_std + 1e-8)

    # Datasets
    train_ds = ASTDatasetCV(X_train_ast_fold, y_train_fold, parts_train_fold, augment=True)
    val_ds = ASTDatasetCV(X_val_ast_fold, y_val_fold, parts_val_fold, augment=False)

    # Balanced sampler
    class_counts = np.bincount(y_train_fold)
    weights = 1.0 / class_counts
    sample_weights = weights[y_train_fold]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    # Fresh FROZEN model
    model_fold = ASTClassifier(num_classes=2, pretrained=True, freeze_base=True).to(device)

    trainable = sum(p.numel() for p in model_fold.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model_fold.parameters())
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    # Only optimize head params - use head LR (5e-4) for all trainable params
    optimizer = torch.optim.AdamW(
        [p for p in model_fold.parameters() if p.requires_grad],
        lr=5e-4, weight_decay=0.01
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-7)

    # Dynamic class weights
    cc = np.bincount(y_train_fold)
    cw = (cc.sum() / (2.0 * cc)).astype(np.float32)
    class_weights_fold = torch.tensor(cw, dtype=torch.float32).to(device)
    criterion = FocalLossCV(alpha=class_weights_fold, gamma=2.0)
    print(f"Class weights: {cw}")

    # Training
    best_score = 0
    best_state = None
    patience, patience_counter = 10, 0

    for epoch in range(30):
        model_fold.train()
        total_loss = 0
        for batch in train_loader:
            inputs = batch['inputs'].to(device)
            labels_batch = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model_fold(inputs)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_fold.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()

        # Evaluate
        part_probs, part_labels_fold, _ = evaluate_fold(model_fold, val_loader, device)

        if len(np.unique(part_labels_fold)) > 1:
            auc = roc_auc_score(part_labels_fold, part_probs)
            fpr, tpr, thresholds = roc_curve(part_labels_fold, part_probs)
            opt_idx = np.argmax(tpr - fpr)
            opt_thresh = thresholds[opt_idx]
            preds_opt = (part_probs >= opt_thresh).astype(int)
            f1_opt = f1_score(part_labels_fold, preds_opt, zero_division=0)
        else:
            auc, f1_opt = 0.5, 0.0

        score = 0.4 * auc + 0.6 * f1_opt
        if score > best_score + 0.01:
            best_score = score
            best_state = copy.deepcopy(model_fold.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    # Load best and get final predictions
    model_fold.load_state_dict(best_state)
    part_probs, part_labels_fold, val_pids = evaluate_fold(model_fold, val_loader, device)

    # Store OOF predictions
    for i, pid in enumerate(val_pids):
        idx = np.where(unique_participants == pid)[0][0]
        frozen_oof_probs[idx] = part_probs[i]

    # Fold metrics
    fold_auc = roc_auc_score(part_labels_fold, part_probs)
    fpr, tpr, thresholds = roc_curve(part_labels_fold, part_probs)
    opt_idx = np.argmax(tpr - fpr)
    opt_thresh = thresholds[opt_idx]
    preds_opt = (part_probs >= opt_thresh).astype(int)

    frozen_fold_results.append({
        'fold': fold + 1,
        'auc': float(fold_auc),
        'f1': float(f1_score(part_labels_fold, preds_opt, zero_division=0)),
        'recall': float(recall_score(part_labels_fold, preds_opt, zero_division=0)),
        'precision': float(precision_score(part_labels_fold, preds_opt, zero_division=0)),
        'threshold': float(opt_thresh),
    })

    print(f"Fold {fold+1}: AUC={fold_auc:.4f}, F1={frozen_fold_results[-1]['f1']:.4f}, "
          f"Rec={frozen_fold_results[-1]['recall']:.4f}, Prec={frozen_fold_results[-1]['precision']:.4f}")

    del model_fold, optimizer, train_ds, val_ds
    torch.cuda.empty_cache()

total_time = time.time() - total_start

# Summary
print("\n" + "="*60)
print("FROZEN AST (LINEAR PROBE) - Per-fold results")
print("="*60)
for r in frozen_fold_results:
    print(f"  Fold {r['fold']}: AUC={r['auc']:.4f}  F1={r['f1']:.4f}  Rec={r['recall']:.4f}  Prec={r['precision']:.4f}")

n = N_FOLDS
t_crit = sp_stats.t.ppf(0.975, df=n-1)

for metric_name in ['auc', 'f1', 'recall', 'precision']:
    vals = [r[metric_name] for r in frozen_fold_results]
    m = np.mean(vals)
    sd = np.std(vals, ddof=1)
    se = sd / np.sqrt(n)
    ci_lo, ci_hi = m - t_crit * se, m + t_crit * se
    print(f"  Mean {metric_name.upper()}: {m:.4f} +/- {sd:.4f}  95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")

# OOF metrics
frozen_oof_auc = roc_auc_score(frozen_oof_labels, frozen_oof_probs)
fpr, tpr, thresholds = roc_curve(frozen_oof_labels, frozen_oof_probs)
opt_idx = np.argmax(tpr - fpr)
frozen_oof_thresh = thresholds[opt_idx]
frozen_oof_preds = (frozen_oof_probs >= frozen_oof_thresh).astype(int)

print(f"\nOOF AUC:       {frozen_oof_auc:.4f}")
print(f"OOF F1:        {f1_score(frozen_oof_labels, frozen_oof_preds, zero_division=0):.4f} (threshold={frozen_oof_thresh:.3f})")
print(f"OOF Recall:    {recall_score(frozen_oof_labels, frozen_oof_preds, zero_division=0):.4f}")
print(f"OOF Precision: {precision_score(frozen_oof_labels, frozen_oof_preds, zero_division=0):.4f}")
print(f"\nTotal CV time: {total_time/60:.1f} minutes")

# Save
np.savez("frozen_ast_baseline_cv_results.npz",
    oof_probs=frozen_oof_probs,
    oof_labels=frozen_oof_labels,
    participant_ids=unique_participants,
    fold_aucs=np.array([r['auc'] for r in frozen_fold_results]),
    fold_f1s=np.array([r['f1'] for r in frozen_fold_results]),
    fold_recalls=np.array([r['recall'] for r in frozen_fold_results]),
    fold_precisions=np.array([r['precision'] for r in frozen_fold_results]),
)
print("Saved: frozen_ast_baseline_cv_results.npz")

---
## 4. E5: Baseline Comparison Table

Compare Fine-tuned AST (from saved results), Frozen AST (linear probe), and Logistic Regression baselines.

In [ ]:
#9 - Load fine-tuned AST results and build comparison table
ft_results = np.load("ast_pd_selected_cv_results.npz", allow_pickle=True)
ft_oof_probs = ft_results['oof_probs']
ft_oof_labels = ft_results['oof_labels']

ft_oof_auc = roc_auc_score(ft_oof_labels, ft_oof_probs)
fpr_ft, tpr_ft, thresholds_ft = roc_curve(ft_oof_labels, ft_oof_probs)
opt_idx_ft = np.argmax(tpr_ft - fpr_ft)
ft_thresh = thresholds_ft[opt_idx_ft]
ft_preds = (ft_oof_probs >= ft_thresh).astype(int)

# Comparison table
print("="*80)
print("BASELINE COMPARISON TABLE - PD Selected Tasks (Participant-Level OOF Metrics)")
print("="*80)
print(f"{'Model':<35} {'OOF AUC':>8} {'OOF F1':>8} {'OOF Recall':>11} {'OOF Prec':>9}")
print("-"*80)

print(f"{'Fine-tuned AST':<35} {ft_oof_auc:>8.3f} "
      f"{f1_score(ft_oof_labels, ft_preds, zero_division=0):>8.3f} "
      f"{recall_score(ft_oof_labels, ft_preds, zero_division=0):>11.3f} "
      f"{precision_score(ft_oof_labels, ft_preds, zero_division=0):>9.3f}")

print(f"{'Frozen AST (linear probe)':<35} {frozen_oof_auc:>8.3f} "
      f"{f1_score(frozen_oof_labels, frozen_oof_preds, zero_division=0):>8.3f} "
      f"{recall_score(frozen_oof_labels, frozen_oof_preds, zero_division=0):>11.3f} "
      f"{precision_score(frozen_oof_labels, frozen_oof_preds, zero_division=0):>9.3f}")

print(f"{'Logistic Regression (spec. stats)':<35} {lr_oof_auc:>8.3f} "
      f"{f1_score(lr_oof_labels, lr_oof_preds, zero_division=0):>8.3f} "
      f"{recall_score(lr_oof_labels, lr_oof_preds, zero_division=0):>11.3f} "
      f"{precision_score(lr_oof_labels, lr_oof_preds, zero_division=0):>9.3f}")

print("-"*80)

# Interpretation
ft_minus_frozen = ft_oof_auc - frozen_oof_auc
ft_minus_lr = ft_oof_auc - lr_oof_auc

print(f"\nFine-tuning gap (Fine-tuned - Frozen):  {ft_minus_frozen:+.3f} AUC")
print(f"Deep learning gap (Fine-tuned - LR):     {ft_minus_lr:+.3f} AUC")
print(f"\nInterpretation:")
if ft_minus_frozen > 0.02:
    print(f"  - Fine-tuning contributes +{ft_minus_frozen:.3f} AUC over frozen representations,")
    print(f"    demonstrating that domain adaptation to clinical speech is beneficial.")
else:
    print(f"  - Frozen and fine-tuned AST perform similarly ({ft_minus_frozen:+.3f} AUC gap),")
    print(f"    suggesting pretrained AudioSet representations transfer well to clinical speech.")

if ft_minus_lr > 0.05:
    print(f"  - The AST outperforms logistic regression by {ft_minus_lr:+.3f} AUC,")
    print(f"    justifying the use of deep learning for this task.")
else:
    print(f"  - The gap over logistic regression is modest ({ft_minus_lr:+.3f} AUC).")

In [ ]:
#10 - Publication-quality ROC comparison figure
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12})

fig, ax = plt.subplots(1, 1, figsize=(7, 6))

# Fine-tuned AST
fpr_ft, tpr_ft, _ = roc_curve(ft_oof_labels, ft_oof_probs)
ax.plot(fpr_ft, tpr_ft, 'b-', linewidth=2, label=f'Fine-tuned AST (AUC={ft_oof_auc:.3f})')

# Frozen AST
fpr_fr, tpr_fr, _ = roc_curve(frozen_oof_labels, frozen_oof_probs)
ax.plot(fpr_fr, tpr_fr, 'r--', linewidth=2, label=f'Frozen AST (AUC={frozen_oof_auc:.3f})')

# Logistic Regression
fpr_lr, tpr_lr, _ = roc_curve(lr_oof_labels, lr_oof_probs)
ax.plot(fpr_lr, tpr_lr, 'g-.', linewidth=2, label=f'Logistic Regression (AUC={lr_oof_auc:.3f})')

# Diagonal
ax.plot([0, 1], [0, 1], 'k:', linewidth=1, alpha=0.5)

ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('PD Selected Tasks: Baseline Comparison (OOF)', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('baseline_roc_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: baseline_roc_comparison.png")

---
## 5. R1: Calibration Analysis

Calibration curves and Brier scores for all three conditions using OOF predictions.

For PD we use the saved fine-tuned results. For Dementia and Depression we regenerate OOF predictions from the saved fold models.

In [ ]:
#11 - Generate Dementia OOF predictions from saved fold models

# Build dementia label
pheno['dementia_label'] = pheno['alz_dementia_mci'].map({'Checked':1, 'Unchecked':0})
dem_labels = pheno[['participant_id','dementia_label']].dropna()
dem_labels['dementia_label'] = dem_labels['dementia_label'].astype(int)

data_dem = spec.merge(dem_labels, on='participant_id', how='inner')
data_dem['time_frames'] = data_dem['spectrogram'].apply(lambda s: np.stack(s).shape[1])

# Same tasks as Dementia notebook
dem_tasks = [
    'Cinderella-Story', 'Productive-Vocabulary-1', 'Productive-Vocabulary-2',
    'Productive-Vocabulary-3', 'Productive-Vocabulary-4', 'Productive-Vocabulary-5',
    'Productive-Vocabulary-6', 'Random-Item-Generation', 'Word-color-Stroop',
]

data_dem_high = data_dem[
    (data_dem['task_name'].isin(dem_tasks)) & (data_dem['time_frames'] >= 100)
].copy()

X_dem_list = []
for _, row in tqdm(data_dem_high.iterrows(), total=len(data_dem_high), desc="Dementia specs"):
    X_dem_list.append(process_spectrogram_raw(row['spectrogram'], 1024))

X_dem_raw = np.stack(X_dem_list)
y_dem_raw = data_dem_high['dementia_label'].values
parts_dem_raw = data_dem_high['participant_id'].values

unique_dem_parts = np.unique(parts_dem_raw)
dem_part_labels = np.array([y_dem_raw[parts_dem_raw == p][0] for p in unique_dem_parts])

print(f"Dementia: {len(X_dem_raw)} recordings, {len(unique_dem_parts)} participants, "
      f"{dem_part_labels.sum()} dementia+")

# Run 5-fold inference with saved models
skf_dem = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dem_oof_probs = np.zeros(len(unique_dem_parts), dtype=np.float32)
dem_oof_labels = dem_part_labels.astype(np.int64).copy()

for fold, (train_idx, val_idx) in enumerate(skf_dem.split(unique_dem_parts, dem_part_labels)):
    print(f"  Dementia fold {fold+1}...")
    val_parts_fold = unique_dem_parts[val_idx]
    train_parts_fold = unique_dem_parts[train_idx]

    train_mask = np.isin(parts_dem_raw, train_parts_fold)
    val_mask = np.isin(parts_dem_raw, val_parts_fold)

    # Resize and normalize using training fold stats
    X_train_resize = np.stack([resize_spectrogram_cv(x) for x in X_dem_raw[train_mask]])
    X_val_resize = np.stack([resize_spectrogram_cv(x) for x in X_dem_raw[val_mask]])
    fold_mean = X_train_resize.mean()
    fold_std = X_train_resize.std()
    X_val_norm = (X_val_resize - fold_mean) / (fold_std + 1e-8)

    val_ds = ASTDatasetCV(X_val_norm, y_dem_raw[val_mask], parts_dem_raw[val_mask], augment=False)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    # Load saved model
    model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=False).to(device)
    model.load_state_dict(torch.load(f"ast_dementia_fold{fold+1}.pt", map_location=device))

    part_probs, part_labels_f, val_pids = evaluate_fold(model, val_loader, device)

    for i, pid in enumerate(val_pids):
        idx = np.where(unique_dem_parts == pid)[0][0]
        dem_oof_probs[idx] = part_probs[i]

    del model
    torch.cuda.empty_cache()

dem_oof_auc = roc_auc_score(dem_oof_labels, dem_oof_probs)
print(f"Dementia OOF AUC: {dem_oof_auc:.4f}")

In [ ]:
#12 - Generate Depression OOF predictions from saved fold models

pheno['depression_label'] = pheno['depression'].map({'Checked':1, 'Unchecked':0})
dep_labels = pheno[['participant_id','depression_label']].dropna()
dep_labels['depression_label'] = dep_labels['depression_label'].astype(int)

data_dep = spec.merge(dep_labels, on='participant_id', how='inner')
data_dep['time_frames'] = data_dep['spectrogram'].apply(lambda s: np.stack(s).shape[1])

# Same tasks as Depression notebook
dep_tasks = ['Animal-fluency', 'Open-response-questions']

data_dep_high = data_dep[
    (data_dep['task_name'].isin(dep_tasks)) & (data_dep['time_frames'] >= 100)
].copy()

X_dep_list = []
for _, row in tqdm(data_dep_high.iterrows(), total=len(data_dep_high), desc="Depression specs"):
    X_dep_list.append(process_spectrogram_raw(row['spectrogram'], 1024))

X_dep_raw = np.stack(X_dep_list)
y_dep_raw = data_dep_high['depression_label'].values
parts_dep_raw = data_dep_high['participant_id'].values

unique_dep_parts = np.unique(parts_dep_raw)
dep_part_labels = np.array([y_dep_raw[parts_dep_raw == p][0] for p in unique_dep_parts])

print(f"Depression: {len(X_dep_raw)} recordings, {len(unique_dep_parts)} participants, "
      f"{dep_part_labels.sum()} depression+")

# Run 5-fold inference with saved models
skf_dep = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dep_oof_probs = np.zeros(len(unique_dep_parts), dtype=np.float32)
dep_oof_labels = dep_part_labels.astype(np.int64).copy()

for fold, (train_idx, val_idx) in enumerate(skf_dep.split(unique_dep_parts, dep_part_labels)):
    print(f"  Depression fold {fold+1}...")
    val_parts_fold = unique_dep_parts[val_idx]
    train_parts_fold = unique_dep_parts[train_idx]

    train_mask = np.isin(parts_dep_raw, train_parts_fold)
    val_mask = np.isin(parts_dep_raw, val_parts_fold)

    X_train_resize = np.stack([resize_spectrogram_cv(x) for x in X_dep_raw[train_mask]])
    X_val_resize = np.stack([resize_spectrogram_cv(x) for x in X_dep_raw[val_mask]])
    fold_mean = X_train_resize.mean()
    fold_std = X_train_resize.std()
    X_val_norm = (X_val_resize - fold_mean) / (fold_std + 1e-8)

    val_ds = ASTDatasetCV(X_val_norm, y_dep_raw[val_mask], parts_dep_raw[val_mask], augment=False)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=False).to(device)
    model.load_state_dict(torch.load(f"ast_depression_fold{fold+1}.pt", map_location=device))

    part_probs, part_labels_f, val_pids = evaluate_fold(model, val_loader, device)

    for i, pid in enumerate(val_pids):
        idx = np.where(unique_dep_parts == pid)[0][0]
        dep_oof_probs[idx] = part_probs[i]

    del model
    torch.cuda.empty_cache()

dep_oof_auc = roc_auc_score(dep_oof_labels, dep_oof_probs)
print(f"Depression OOF AUC: {dep_oof_auc:.4f}")

In [ ]:
#13 - Calibration curves and Brier scores
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

conditions = [
    ("a) Parkinson's Disease", ft_oof_labels, ft_oof_probs),
    ("b) Dementia", dem_oof_labels, dem_oof_probs),
    ("c) Depression", dep_oof_labels, dep_oof_probs),
]

for ax, (title, y_true, y_prob) in zip(axes, conditions):
    brier = brier_score_loss(y_true, y_prob)
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=8, strategy='uniform')

    ax.plot(prob_pred, prob_true, 'bo-', linewidth=2, markersize=6, label='Model')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Perfectly calibrated')
    ax.set_xlabel('Mean predicted probability', fontsize=12)
    ax.set_ylabel('Fraction of positives', fontsize=12)
    ax.set_title(f'{title}\nBrier score = {brier:.3f}', fontsize=13)
    ax.legend(loc='upper left', fontsize=10)
    ax.set_xlim([-0.05, 1.05])
    ax.set_ylim([-0.05, 1.05])
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('calibration_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nBrier Scores:")
for title, y_true, y_prob in conditions:
    brier = brier_score_loss(y_true, y_prob)
    print(f"  {title}: {brier:.4f}")

print("\nSaved: calibration_curves.png")

---
## 6. R2: Per-Task Contribution Analysis

Disaggregate PD selected-tasks OOF predictions by task to show which speech tasks contribute most to PD discrimination.

In [ ]:
#14 - Per-task recording-level AUC analysis
#
# We need recording-level OOF predictions mapped back to tasks.
# The saved OOF are participant-level. To get recording-level predictions
# we re-run fold-by-fold inference and keep the recording-level probs.

skf_task = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rec_oof_probs = np.full(len(X_raw), np.nan, dtype=np.float32)
rec_oof_labels = y_raw.copy()
rec_oof_tasks = task_names_raw.copy()

for fold, (train_idx, val_idx) in enumerate(skf_task.split(unique_participants, participant_labels)):
    print(f"  Per-task fold {fold+1}...")
    val_parts_fold = unique_participants[val_idx]
    train_parts_fold = unique_participants[train_idx]

    train_mask = np.isin(participants_raw, train_parts_fold)
    val_mask = np.isin(participants_raw, val_parts_fold)

    # Resize and normalize using training fold stats
    X_train_resize = np.stack([resize_spectrogram_cv(x) for x in X_raw[train_mask]])
    X_val_resize = np.stack([resize_spectrogram_cv(x) for x in X_raw[val_mask]])
    fold_mean = X_train_resize.mean()
    fold_std = X_train_resize.std()
    X_val_norm = (X_val_resize - fold_mean) / (fold_std + 1e-8)

    val_ds = ASTDatasetCV(X_val_norm, y_raw[val_mask], participants_raw[val_mask], augment=False)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    # Load saved fine-tuned model
    model = ASTClassifier(num_classes=2, pretrained=True, freeze_base=False).to(device)
    model.load_state_dict(torch.load(f"ast_pd_selected_fold{fold+1}.pt", map_location=device))
    model.eval()

    # Get recording-level predictions
    all_probs = []
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch['inputs'].to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)

    # Map back to original indices
    val_indices = np.where(val_mask)[0]
    for i, orig_idx in enumerate(val_indices):
        rec_oof_probs[orig_idx] = all_probs[i]

    del model
    torch.cuda.empty_cache()

# Verify all recordings got predictions
assert not np.any(np.isnan(rec_oof_probs)), "Some recordings missing OOF predictions!"
print(f"\nAll {len(rec_oof_probs)} recordings have OOF predictions.")

In [ ]:
#15 - Per-task AUC bar chart

task_results = []
for task in high_pd_only_tasks:
    mask = rec_oof_tasks == task
    n_recs = mask.sum()
    n_pd = rec_oof_labels[mask].sum()
    n_non_pd = n_recs - n_pd

    if len(np.unique(rec_oof_labels[mask])) > 1:
        task_auc = roc_auc_score(rec_oof_labels[mask], rec_oof_probs[mask])
    else:
        task_auc = float('nan')

    task_results.append({
        'task': task,
        'n_recordings': int(n_recs),
        'n_pd': int(n_pd),
        'n_non_pd': int(n_non_pd),
        'recording_auc': task_auc,
    })
    print(f"{task:<30} n={n_recs:>4} (PD={n_pd:>3}, Non-PD={n_non_pd:>3})  "
          f"Recording-level AUC={task_auc:.3f}" if not np.isnan(task_auc) else
          f"{task:<30} n={n_recs:>4} (PD={n_pd:>3}, Non-PD={n_non_pd:>3})  AUC=N/A (single class)")

# Bar chart
valid_tasks = [r for r in task_results if not np.isnan(r['recording_auc'])]
valid_tasks.sort(key=lambda x: x['recording_auc'], reverse=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [2, 1]})

task_names_short = [t['task'].replace('Productive-Vocabulary-', 'PV-') for t in valid_tasks]
aucs_plot = [t['recording_auc'] for t in valid_tasks]
counts = [t['n_recordings'] for t in valid_tasks]

colors = plt.cm.RdYlGn([(a - 0.4) / 0.6 for a in aucs_plot])

bars1 = ax1.bar(range(len(valid_tasks)), aucs_plot, color=colors, edgecolor='black', linewidth=0.5)
ax1.axhline(y=0.5, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Chance (0.5)')
ax1.set_ylabel('Recording-level AUC', fontsize=12)
ax1.set_title('Per-Task Contribution to PD Classification (OOF Predictions)', fontsize=14)
ax1.set_xticks(range(len(valid_tasks)))
ax1.set_xticklabels(task_names_short, rotation=45, ha='right', fontsize=10)
for i, v in enumerate(aucs_plot):
    ax1.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_ylim([0, max(aucs_plot) + 0.08])
ax1.grid(axis='y', alpha=0.3)

bars2 = ax2.bar(range(len(valid_tasks)), counts, color='steelblue', edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Number of recordings', fontsize=12)
ax2.set_xticks(range(len(valid_tasks)))
ax2.set_xticklabels(task_names_short, rotation=45, ha='right', fontsize=10)
for i, v in enumerate(counts):
    ax2.text(i, v + 1, str(v), ha='center', va='bottom', fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('per_task_contribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: per_task_contribution.png")

---
## 7. R3: Comorbidity Quantification

Cross-tabulate PD and Dementia labels for shared participants.

In [ ]:
#16 - Comorbidity analysis

pd_labels_full = pheno[['participant_id', 'parkinsons']].dropna()
dem_labels_full = pheno[['participant_id', 'alz_dementia_mci']].dropna()
dep_labels_full = pheno[['participant_id', 'depression']].dropna()

# PD-Dementia overlap
pd_dem = pd_labels_full.merge(dem_labels_full, on='participant_id', how='inner')
print("="*60)
print("PD-DEMENTIA COMORBIDITY (all labeled participants)")
print("="*60)
print(f"Participants with both PD and Dementia labels: {len(pd_dem)}")

both_pos = ((pd_dem['parkinsons']=='Checked') & (pd_dem['alz_dementia_mci']=='Checked')).sum()
pd_only = ((pd_dem['parkinsons']=='Checked') & (pd_dem['alz_dementia_mci']=='Unchecked')).sum()
dem_only = ((pd_dem['parkinsons']=='Unchecked') & (pd_dem['alz_dementia_mci']=='Checked')).sum()
neither = ((pd_dem['parkinsons']=='Unchecked') & (pd_dem['alz_dementia_mci']=='Unchecked')).sum()

print(f"\n  PD+ and Dementia+: {both_pos}")
print(f"  PD+ and Dementia-: {pd_only}")
print(f"  PD- and Dementia+: {dem_only}")
print(f"  PD- and Dementia-: {neither}")

# Cross-tab
print("\nCross-tabulation:")
ct = pd.crosstab(
    pd_dem['parkinsons'].map({'Checked': 'PD+', 'Unchecked': 'PD-'}),
    pd_dem['alz_dementia_mci'].map({'Checked': 'Dementia+', 'Unchecked': 'Dementia-'}),
    margins=True
)
print(ct)

# PD-Depression overlap
pd_dep = pd_labels_full.merge(dep_labels_full, on='participant_id', how='inner')
print(f"\n{'='*60}")
print("PD-DEPRESSION COMORBIDITY")
print(f"{'='*60}")
print(f"Participants with both PD and Depression labels: {len(pd_dep)}")

pd_dep_both = ((pd_dep['parkinsons']=='Checked') & (pd_dep['depression']=='Checked')).sum()
pd_dep_pd_only = ((pd_dep['parkinsons']=='Checked') & (pd_dep['depression']=='Unchecked')).sum()
pd_dep_dep_only = ((pd_dep['parkinsons']=='Unchecked') & (pd_dep['depression']=='Checked')).sum()
pd_dep_neither = ((pd_dep['parkinsons']=='Unchecked') & (pd_dep['depression']=='Unchecked')).sum()

print(f"\n  PD+ and Depression+: {pd_dep_both}")
print(f"  PD+ and Depression-: {pd_dep_pd_only}")
print(f"  PD- and Depression+: {pd_dep_dep_only}")
print(f"  PD- and Depression-: {pd_dep_neither}")

ct2 = pd.crosstab(
    pd_dep['parkinsons'].map({'Checked': 'PD+', 'Unchecked': 'PD-'}),
    pd_dep['depression'].map({'Checked': 'Depression+', 'Unchecked': 'Depression-'}),
    margins=True
)
print(ct2)

# Dementia-Depression overlap
dem_dep = dem_labels_full.merge(dep_labels_full, on='participant_id', how='inner')
print(f"\n{'='*60}")
print("DEMENTIA-DEPRESSION COMORBIDITY")
print(f"{'='*60}")

dd_both = ((dem_dep['alz_dementia_mci']=='Checked') & (dem_dep['depression']=='Checked')).sum()
dd_dem_only = ((dem_dep['alz_dementia_mci']=='Checked') & (dem_dep['depression']=='Unchecked')).sum()
dd_dep_only = ((dem_dep['alz_dementia_mci']=='Unchecked') & (dem_dep['depression']=='Checked')).sum()
dd_neither = ((dem_dep['alz_dementia_mci']=='Unchecked') & (dem_dep['depression']=='Unchecked')).sum()

print(f"  Dementia+ and Depression+: {dd_both}")
print(f"  Dementia+ and Depression-: {dd_dem_only}")
print(f"  Dementia- and Depression+: {dd_dep_only}")
print(f"  Dementia- and Depression-: {dd_neither}")

---
## Summary

All results generated by this notebook:

| File | Description |
|---|---|
| `lr_baseline_cv_results.npz` | Logistic regression OOF predictions and fold metrics |
| `frozen_ast_baseline_cv_results.npz` | Frozen AST OOF predictions and fold metrics |
| `baseline_roc_comparison.png` | ROC curves comparing all three models |
| `calibration_curves.png` | Calibration curves for PD, Dementia, Depression |
| `per_task_contribution.png` | Per-task recording-level AUC and counts |